In [1]:

import os, json, numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.svm import SVC
# Ensure qml-benchmark/src is on PYTHONPATH so `from src import ...` works
import sys, pathlib, os
try:
    p = pathlib.Path.cwd()
except Exception:
    # fallback to known workspace location inside the devcontainer
    p = pathlib.Path('/workspaces/Quantum-machine-learning-/qml-benchmark/notebooks')
# If running from notebooks/, parent is qml-benchmark/
candidate = p.parent
if (candidate / 'src').exists():
    sys.path.insert(0, str(candidate))
else:
    # fallback: try repository root / qml-benchmark
    candidate2 = p.parent.parent / 'qml-benchmark'
    if (candidate2 / 'src').exists():
        sys.path.insert(0, str(candidate2))

from src.kernels_qiskit import build_quantum_kernel, rescale, gram_train, gram_test
from src.cv_utils import make_cv, param_iter, write_json, mean_std

SEED = 42
np.random.seed(SEED)

RESULTS_DIR = str((candidate / 'results' / 'qsvc_cv'))
os.makedirs(RESULTS_DIR, exist_ok=True)

X = np.random.randn(200, 2)
y = ((X[:,0] * X[:,1]) > 0).astype(int)

grid = {"reps": [1, 2], "scale": [0.5, 1.0], "C": [0.1, 1.0, 10.0]}
cv = make_cv(n_splits=5, seed=SEED)

fold_results = []
best_score, best_cfg = -np.inf, None

for cfg in param_iter(grid):
    accs, aucs = [], []
    for fold, (tr, te) in enumerate(cv.split(X, y), start=1):
        X_tr, X_te = X[tr], X[te]
        y_tr, y_te = y[tr], y[te]

        qk = build_quantum_kernel(feature_dimension=X.shape[1], reps=cfg["reps"])
        X_tr_s, X_te_s = rescale(X_tr, cfg["scale"]), rescale(X_te, cfg["scale"])

        # QSVC path
        from qiskit_machine_learning.algorithms import QSVC
        qsvc = QSVC(quantum_kernel=qk, C=cfg["C"])
        qsvc.fit(X_tr_s, y_tr)
        y_pred = qsvc.predict(X_te_s)
        acc = accuracy_score(y_te, y_pred)
        try:
            auc = roc_auc_score(y_te, y_pred)
        except Exception:
            auc = np.nan

        # Export Gram matrices
        K_tr = gram_train(qk, X_tr_s)
        K_te = gram_test(qk, X_tr_s, X_te_s)
        np.save(os.path.join(RESULTS_DIR, f"K_train_f{fold}_r{cfg['reps']}_s{cfg['scale']}.npy"), K_tr)
        np.save(os.path.join(RESULTS_DIR, f"K_test_f{fold}_r{cfg['reps']}_s{cfg['scale']}.npy"), K_te)

        # Precomputed SVC parity: ensure shapes match sklearn expectations
        K_tr = np.asarray(K_tr)
        K_te = np.asarray(K_te)
        # Ensure shapes are (n_train, n_train) and (n_test, n_train)
        if K_tr.shape[0] != len(y_tr) and K_tr.shape[1] == len(y_tr):
            K_tr = K_tr.T
        if K_te.ndim == 2 and K_te.shape[1] != K_tr.shape[0] and K_te.shape[0] == K_tr.shape[0]:
            K_te = K_te.T

        svc_pre = SVC(kernel="precomputed", C=cfg["C"])
        svc_pre.fit(K_tr, y_tr)
        y_pred_pre = svc_pre.predict(K_te)
        acc_pre = accuracy_score(y_te, y_pred_pre)

        accs.append(acc); aucs.append(auc)

    stats = {"cfg": cfg, "acc": mean_std(accs), "auc": mean_std(aucs)}
    fold_results.append(stats)
    if stats["acc"]["mean"] > best_score:
        best_score, best_cfg = stats["acc"]["mean"], cfg

write_json(os.path.join(RESULTS_DIR, "cv_results.json"), fold_results)
write_json(os.path.join(RESULTS_DIR, "best_cfg.json"), {"best_cfg": best_cfg, "best_acc": best_score})
print("Best:", best_cfg, "Acc:", best_score)


Best: {'C': 10.0, 'reps': 1, 'scale': 1.0} Acc: 0.925
